Import PDF Document :

In [1]:
import os 
import requests

pdf_path = "knowledge_base/artificial_intelligence_technology.pdf"

if not os.path.exists(pdf_path):
  print(f"[Info] file doesn't exist, downloading... ")

  #Enter the url of the pdf 
  url = "https://link.springer.com/content/pdf/10.1007/978-981-19-2879-6.pdf"

  # Local filename to save the downloaded file 
  file_name = pdf_path

  # Send a GET request to the url 
  response = requests.get(url , timeout=5)

  # check if the request was successful
  if response.status_code == 200:
    # Open the file and save it 
    with open (file_name,"wb") as  file:
      file.write(response.content)
    print(f"[Info] the file has been downloaded and saved as {file_name}")

  else:
    print(f"[Info] Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} Exists.")


File knowledge_base/artificial_intelligence_technology.pdf Exists.


Ingestion / Open the pdf and extract the text and metadata from the pdf:

In [11]:
import fitz
from tqdm.auto import tqdm

def text_format(text:str) -> str:
  """Performs minor formatting on text."""
  clear_text = text.replace("\n", " ").strip()
  # Other potential text formatting functions can go here
  return clear_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path : str)-> list[dict]:
  """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
  doc = fitz.open(pdf_path) # Open a document
  pages_and_texts = []
  for page_number,page in tqdm(enumerate(doc,start=1)):
    text = page.get_text()
    # text = text_format(text)
    if not text:
      continue

    # pages_and_texts.append({"page_number": page_number - 13,  # adjust page numbers since our PDF starts on page 14
    #                             "page_char_count": len(text),
    #                             "page_word_count": len(text.split(" ")),
    #                             "page_sentence_count_raw": len(text.split(". ")),
    #                             "page_token_count": len(text) / 4,  # 1 token = ~4 chars,
    #                             "text": text})
    pages_and_texts.append({"text": text,
                            "metadata":{
                              "source": pdf_path,
                              "pdf_page": page_number,
                              "page_label": page.get_label()
                            }})
  return pages_and_texts
pages_and_texts = open_and_read_pdf("knowledge_base/artificial_intelligence_technology.pdf")
print(len(pages_and_texts))
print(pages_and_texts[0])
print(pages_and_texts[13])
print(pages_and_texts[-1])
print(pages_and_texts[18])
print(pages_and_texts[31])

0it [00:00, ?it/s]

308it [00:01, 179.71it/s]


308
{'text': 'Official Textbooks for Huawei ICT Academy\nARTIFICIAL \nINTELLIGENCE \nTECHNOLOGY\nHuawei Technologies Co., Ltd.\n', 'metadata': {'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 1, 'page_label': 'C1'}}
{'text': 'Chapter 1\nA General Introduction to Artiﬁcial\nIntelligence\nThe emergence and rise of artiﬁcial intelligence undoubtedly played an important\nrole during the development of the Internet. Over the past decade, with extensive\napplications in the society, artiﬁcial intelligence has become more relevant to\npeople’s daily life. This chapter introduces the concept of artiﬁcial intelligence, the\nrelated technologies, and the existing controversies over the topic.\n1.1\nThe Concept of Artiﬁcial Intelligence\n1.1.1\nWhat Is Artiﬁcial Intelligence?\nCurrently, people mainly learn about artiﬁcial intelligence (AI) through news,\nmovies, and the applications in daily life, as shown by Fig. 1.1.\nA rather widely accepted deﬁnition of AI, als

In [12]:
doc = fitz.open("knowledge_base/artificial_intelligence_technology.pdf")

page = doc[31]

data = page.get_text("dict")

for block_no, block in enumerate(data["blocks"]):

    if "lines" not in block:
        continue

    print(f"\n========== BLOCK {block_no} ==========")

    for line in block["lines"]:

        line_text = ""

        for span in line["spans"]:
            line_text += span["text"]

        print(
            f"y={line['bbox'][1]:.1f} "
            f"→ {repr(line_text)}"
        )

        for span in line["spans"]:
            print(
                f"    text={repr(span['text'])} | "
                f"size={span['size']} | "
                f"font={span['font']} | "
                f"flags={span['flags']}"
            )


========== BLOCK 0 ==========
y=225.6 → 'GPUs are good at dealing with operations that are intensive and easy to be run in'
    text='GPUs are good at dealing with operations that are intensive and easy to be run in' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
y=237.6 → 'parallel, while CPUs excel at logic control and serial operations.'
    text='parallel, while CPUs excel at logic control and serial operations.' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4

========== BLOCK 1 ==========
y=249.6 → 'The difference in architecture between GPU and CPU is because that they'
    text='The' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
    text=' ' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
    text='difference' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
    text=' ' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
    text='in' | size=9.962599754333496 | font=AdvOTab62ddd12 | flags=4
    text=' ' | size=9.962599754333

In [3]:
doc = fitz.open("knowledge_base/artificial_intelligence_technology.pdf")

print("Total PDF pages:",len(doc))
print("Extracted pages:",len(pages_and_texts))

Total PDF pages: 308
Extracted pages: 308


In [5]:
# print(pages_and_texts[0]["metadata"])
# print(pages_and_texts[12]["metadata"])
# print(pages_and_texts[13]["metadata"])
# print(pages_and_texts[-1]["metadata"])

Get some stats on the text

In [6]:
import pandas as pd 

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-12,108,15,1,27.00,Official Textbooks for Huawei ICT Academy ARTI...
1,-11,33,3,1,8.25,Artiﬁcial Intelligence Technology
2,-10,63,7,2,15.75,"Huawei Technologies Co., Ltd. Artiﬁcial Intell..."
3,-9,2976,445,20,744.00,"Huawei Technologies Co., Ltd. Hangzhou, China ..."
4,-8,2389,377,14,597.25,Preface The rapid development of information t...


In [8]:
df.describe()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,308.000000,308.00000,308.000000,308.000000,308.000000
mean,141.500000,1908.36039,307.519481,32.775974,477.090097
std,89.056162,747.83511,160.649850,122.403369,186.958778
min,-12.000000,33.00000,3.000000,1.000000,8.250000
25%,64.750000,1478.75000,221.750000,13.000000,369.687500
50%,141.500000,1878.50000,294.500000,16.000000,469.625000
75%,218.250000,2478.25000,388.000000,21.000000,619.562500
max,295.000000,3747.00000,1392.000000,1183.000000,936.750000
